# Experiment 2 (6-fold leave-one-event-out) · Colab GPU training

Based on `docs/PLAN.md` v10 §7. **One fold per session.** Outputs are written to Google Drive and can be resumed after a disconnect.

**Drive directory layout** (generated locally by `code/41_pack_modelB_data.py pack`, then uploaded as a whole):
```
MyDrive/Flood2/
├── modelB_v1.tar            data package (2.6 GB)
├── modelB_v1.tar.md5
├── modelB_v1_manifest.csv   per-file md5
├── code/                    32 33 35 38 39 40 41 + this notebook
└── models/exp2/fold_<event>/  training outputs (created automatically)
```

**Order**: ① settings → ② mount Drive → ③ prepare code and data → ④ check GPU → ⑤ smoke test (the first time in each new session) → ⑥ start / resume training → ⑦ view progress → ⑧ evaluate → ⑨ record GPU and runtime

**Resume after disconnect**: after reconnecting, run ① ② ③ ④ ⑥ in order (⑥ detects the checkpoint automatically and adds `--resume`), then use ⑦ to view progress. At most the unfinished epoch is lost.

Runtime type: menu "Runtime → Change runtime type → GPU".

## ① Settings (only change FOLD each time)

In [ ]:
FOLD = 'matthew'   # held-out event, one of six: irma / matthew / ian / milton / dorian / beryl; one fold per session

DRIVE_ROOT = '/content/drive/MyDrive/Flood2'
PROJ       = '/content/Flood_2.0'                  # local working directory (code and extracted data)
DATA_DIR   = f'{PROJ}/data/modelB_v1'
OUT_ROOT   = f'{DRIVE_ROOT}/models'                # training outputs go to Drive, so they survive disconnects
FOLD_DIR   = f'{OUT_ROOT}/exp2/fold_{FOLD}'
assert FOLD in ['irma', 'matthew', 'ian', 'milton', 'dorian', 'beryl'], FOLD
print('FOLD =', FOLD, '\nOutput directory =', FOLD_DIR)

## ② Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for f in ['modelB_v1.tar', 'modelB_v1.tar.md5', 'modelB_v1_manifest.csv', 'code/38_exp2_train.py']:
    p = f'{DRIVE_ROOT}/{f}'; print(('OK  ' if os.path.exists(p) else 'MISSING ') + p)

## ③ Prepare code and data (copy → extract → md5 check; skipped automatically if already done in this session)

In [ ]:
import os, shutil, subprocess, time
os.makedirs(f'{PROJ}/code', exist_ok=True); os.makedirs(f'{PROJ}/data', exist_ok=True)
for f in os.listdir(f'{DRIVE_ROOT}/code'):
    shutil.copy2(f'{DRIVE_ROOT}/code/{f}', f'{PROJ}/code/{f}')
print('Code copied:', sorted(os.listdir(f'{PROJ}/code')))

ok_flag = f'{DATA_DIR}/.verified'
if os.path.exists(ok_flag):
    print('Data already extracted and verified, skipping')
else:
    t0 = time.time()
    shutil.copy2(f'{DRIVE_ROOT}/modelB_v1.tar', '/content/modelB_v1.tar')
    print('tar copied in %.0f s' % (time.time() - t0))
    tar_md5 = subprocess.run(['md5sum', '/content/modelB_v1.tar'], capture_output=True, text=True).stdout.split()[0]
    expect = open(f'{DRIVE_ROOT}/modelB_v1.tar.md5').read().split()[0]
    assert tar_md5 == expect, ('tar md5 mismatch', tar_md5, expect)
    print('tar md5 matches', tar_md5)
    subprocess.run(['tar', '-xf', '/content/modelB_v1.tar', '-C', f'{PROJ}/data'], check=True)
    os.remove('/content/modelB_v1.tar')
    r = subprocess.run(['python3', f'{PROJ}/code/41_pack_modelB_data.py', 'verify', '--data-dir', DATA_DIR,
                        '--manifest', f'{DRIVE_ROOT}/modelB_v1_manifest.csv'], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    assert r.returncode == 0, 'per-file md5 verification failed, stopping'
    open(ok_flag, 'w').write('ok\n')
    print('Extraction + verification done, %.0f s in total' % (time.time() - t0))

## ④ Check GPU and torch version

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, platform
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| python', platform.python_version())
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU: change the runtime type to GPU and rerun from ①')

## ⑤ Smoke test (small data: train 3 runs / validate 1 run / 2 epochs, output in /content/smoke, nothing written to Drive)

In [ ]:
import shutil; shutil.rmtree('/content/smoke', ignore_errors=True)
!cd {PROJ} && python3 -u code/38_exp2_train.py --fold {FOLD} --data-dir {DATA_DIR} --out-root /content/smoke --smoke 2>&1 | tail -n 25
print('---- resume check: raise the limit to 3 epochs and --resume ----')
!cd {PROJ} && python3 -u code/38_exp2_train.py --fold {FOLD} --data-dir {DATA_DIR} --out-root /content/smoke --smoke --epochs 3 --resume 2>&1 | grep -E "resume|epoch 3 / |training finished|Error|Traceback"
!cd {PROJ} && python3 -u code/39_exp2_eval.py --fold {FOLD} --data-dir {DATA_DIR} --out-root /content/smoke --smoke 2>&1 | tail -n 4

Expected: the first command runs 2 epochs and saves a checkpoint; the second prints the resume line (resuming from epoch 3) and finishes epoch 3; the third prints single-step / rollout RMSE for 1 run. If there is **no Traceback and the loss is finite**, continue (smoke-test numbers are not conclusions).

## ⑥ Start / resume training for this fold (runs in the background; auto --resume if a checkpoint exists; not started again if already running)

In [ ]:
import os, subprocess, shlex
os.makedirs(FOLD_DIR, exist_ok=True)
running = subprocess.run(['pgrep', '-f', f'[3]8_exp2_train.py --fold {FOLD} '], capture_output=True, text=True).stdout.split()
if running:
    print('Training for this fold is already running, PID:', running)
else:
    resume = os.path.exists(f'{FOLD_DIR}/checkpoint_last.pt')
    cmd = (f'cd {shlex.quote(PROJ)} && nohup python3 -u code/38_exp2_train.py --fold {FOLD} '
           f'--data-dir {shlex.quote(DATA_DIR)} --out-root {shlex.quote(OUT_ROOT)} ' + ('--resume ' if resume else '') +
           f'>> {shlex.quote(FOLD_DIR + "/train_stdout.log")} 2>&1 &')
    os.system(cmd)
    print(('Resumed' if resume else 'Started fresh') + ':', cmd)
    print('Run ⑦ in a few tens of seconds to view progress; do not close this page (closing or disconnecting stops the background training)')

## ⑦ View progress (refreshes every 60 s, 60 times; you can interrupt this cell at any time without affecting training)

In [ ]:
%run {PROJ}/code/40_watch_colab.py --fold {FOLD} --out-root {OUT_ROOT} --interval 60 --count 60

In [ ]:
# show once + whether the process is still alive
!pgrep -fl "[3]8_exp2_train.py --fold {FOLD} " || echo "not running (finished / early-stopped / disconnected)"
%run {PROJ}/code/40_watch_colab.py --fold {FOLD} --out-root {OUT_ROOT}

## ⑧ Evaluate (after training): the held-out event's 10 test runs, single-step + rollout, all 168 time steps

In [ ]:
!pgrep -fl "[3]8_exp2_train.py --fold {FOLD} " && echo "!! training still running, evaluate after it finishes" || (cd {PROJ} && python3 -u code/39_exp2_eval.py --fold {FOLD} --data-dir {DATA_DIR} --out-root {OUT_ROOT})
# to evaluate all 81 runs of the held-out event, add --all-runs

## ⑨ Record GPU model and measured runtime (copy the results into docs/PROGRESS.md)

In [ ]:
import json, csv, re
info = json.load(open(f'{FOLD_DIR}/fold_info.json'))
for s in info['sessions']:
    print('session', s['started_at'], '| GPU', s.get('gpu', s['device']), '| torch', s['torch'], '| from epoch', s['start_epoch'], '(resumed)' if s['resume'] else '')
rows = list(csv.DictReader(open(f'{FOLD_DIR}/train_log.csv')))
if rows:
    secs = [float(r['epoch_sec']) for r in rows]
    best = min(rows, key=lambda r: float(r['val_loss']))
    print('%d epochs completed, %.2f h in total, %.0f s/epoch on average; best epoch %s val %s' % (len(rows), sum(secs) / 3600, sum(secs) / len(secs), best['epoch'], best['val_loss']))
sps = [float(m.group(1)) for m in re.finditer(r'\| ([\d.]+) s/step', open(f'{FOLD_DIR}/train_stdout.log').read())]
if sps: print('s/step: median %.3f (%d progress lines)' % (sorted(sps)[len(sps) // 2], len(sps)))